In [1]:
import time
import sys
import os
import time
import threading
import subprocess
import requests
from openai import OpenAI
notebook_start = time.perf_counter()

# ---- Time budget ------------------------------------------------------------
# Started here, on the first executed cell: every later stage measures against it.
RUN_START = time.time()
TIME_BUDGET_S = 12 * 3600
TIME_MARGIN_S = 60*5              # slack so the notebook always finishes cleanly
DEADLINE = RUN_START + TIME_BUDGET_S - TIME_MARGIN_S

def time_left():
    return DEADLINE - time.time()


def out_of_time():
    return time_left() <= 0

In [2]:
os.environ["CUDA_PATH"] = "/usr/local/cuda/bin/nvcc"

In [3]:
os.environ["CUDA_PATH"] = "/usr/local/cuda/bin/nvcc"
os.environ["LD_LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + os.environ.get("LIBRARY_PATH", "")
os.environ["VLLM_LOG_STATS_INTERVAL"] = "2"

os.environ["VLLM_LOG_STATS_INTERVAL"] = str(30)

In [4]:
# !rm -rf /kaggle/tmp/setup/wheels

In [5]:
%%bash

INPUT_ARCHIVE="/kaggle/input/notebooks/laureanoarcanio/utils-vllm-v0-27-1/wheels.tar.gz"
TEMP_DIR="/kaggle/tmp/setup"

mkdir -p "$TEMP_DIR"

if [ ! -d "$TEMP_DIR/wheels" ]; then
    tar -xzf "$INPUT_ARCHIVE" -C "$TEMP_DIR"
fi

if [ -d "$TEMP_DIR/wheels" ]; then
    WHEEL_DIR="$TEMP_DIR/wheels"
else
    WHEEL_DIR="$TEMP_DIR"
fi

echo "Using wheel dir: $WHEEL_DIR"
ls -lah "$WHEEL_DIR" | head -50

python -m pip install \
    --no-index \
    --find-links "$WHEEL_DIR" \
    uv

uv pip install \
    --torch-backend=auto \
    --no-index \
    --find-links "$WHEEL_DIR" \
    vllm

python - <<'PY'
import torch, vllm
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())
print("vllm:", vllm.__version__)
PY

Using wheel dir: /kaggle/tmp/setup/wheels
total 4.2G
drwxr-xr-x 2 root root  20K Oct  7 12:54 .
drwxr-xr-x 3 root root 4.0K Oct 16 15:24 ..
-rw-r--r-- 1 root root 386K Oct  7 12:54 accelerate-1.15.0-py3-none-any.whl
-rw-r--r-- 1 root root 9.2K Oct  7 12:54 agent_detector-2.0.0-py3-none-any.whl
-rw-r--r-- 1 root root  15K Oct  7 12:54 aiohappyeyeballs-2.7.1-py3-none-any.whl
-rw-r--r-- 1 root root 1.9M Oct  7 12:54 aiohttp-3.14.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
-rw-r--r-- 1 root root 7.4K Oct  7 12:54 aiosignal-1.4.0-py3-none-any.whl
-rw-r--r-- 1 root root 5.2K Oct  7 12:54 annotated_doc-0.0.5-py3-none-any.whl
-rw-r--r-- 1 root root  14K Oct  7 12:54 annotated_types-0.8.0-py3-none-any.whl
-rw-r--r-- 1 root root 1.7M Oct  7 12:54 anthropic-1.11.0-py3-none-any.whl
-rw-r--r-- 1 root root 129K Oct  7 12:54 anyio-4.15.1-py3-none-any.whl
-rw-r--r-- 1 root root 2.6M Oct  7 12:54 apache_tvm_ffi-0.1.11-cp312-abi3-manylinux_2_24_x86_64.manylinux_2_2

Using Python 3.12.13 environment at: /usr
Checked 1 package in 135ms


In [6]:
import json

class VLLM_SERVER:
    served_model_name = "qwen3.8"
    model_path = "/kaggle/input/models/laureanoarcanio/swift-qwen3-8-27b-awq-int4/pytorch/default/1/Swift-Qwen3.8-27B-AWQ-INT4"

    host = "0.0.0.0"
    workers = 12
    tensor_parallel_size = 2
    max_model_len = 96*1024
    max_num_seqs = 8
    gpu_memory_utilization = 0.95
    server_timeout = 800 * 2
    session_timeout = 960

    def __init__(self, port: int = 5001, gpus: str = "0,1", wait: bool = True):
        self.port = port
        self.gpus = gpus
        self.base_url = f"http://127.0.0.1:{port}/v1"
        self.api_key = "sk-local"

        self.client = OpenAI(
            base_url=self.base_url,
            api_key=self.api_key,
            timeout=self.session_timeout,
        )

        self.server_process = self._start_server()
        self._start_log_stream()
        if wait:
            self.wait_for_server()

    def _build_command(self) -> list[str]:
        return [
            sys.executable,
            "-m",
            "vllm.entrypoints.openai.api_server",
            "--model", self.model_path,
            "--served-model-name", self.served_model_name,
            "--host", self.host,
            "--port", str(self.port),
            "--tensor-parallel-size", str(self.tensor_parallel_size),
            "--max-model-len", str(self.max_model_len),
            "--max-num-seqs", str(self.max_num_seqs),
            "--max-num-batched-tokens", "8192",
            "--performance-mode", "throughput",
            "--gpu-memory-utilization", str(self.gpu_memory_utilization),
            "--kv-cache-dtype", "fp8",
            "--enable-prefix-caching",
            "--enable-chunked-prefill",
            "--mamba-ssm-cache-dtype", "bfloat16",
            "--reasoning-parser", "qwen3",
            "--enable-auto-tool-choice",
            "--tool-call-parser", "qwen3_coder",
            "--default-chat-template-kwargs", '{"enable_thinking": true}',
            "--language-model-only",
            # "--scheduling-policy", "priority",
            # "--kv-offloading-size", "24"
            # "--speculative-config", json.dumps({
            #     "method": "dflash",
            #     "model": "/kaggle/input/models/laureanoarcanio/qwen3-8-27b-dflash2/pytorch/default/1/incoai/Qwen3.8-27B-DFlash2",
            #     "num_speculative_tokens": 7,
            # }),
            "--kv-cache-memory", str(8_000_000_000),
            "--speculative-config", json.dumps({
                "method": "mtp",
                "num_speculative_tokens": 2,
            }),
            "--watermark", "0.03",
        ]

    def _start_server(self) -> subprocess.Popen:
        cuda_home = os.path.join(os.getcwd(), ".cuda13")
        if os.path.isdir(cuda_home):
            os.environ.setdefault("CUDA_HOME", cuda_home)

        # Own GPUs and own compile cache: instances boot in parallel and must not race
        # on the same torch.compile cache directory.
        env = dict(os.environ,
                   CUDA_VISIBLE_DEVICES=self.gpus,
                   VLLM_CACHE_ROOT=os.path.expanduser(f"~/.cache/vllm-{self.port}"))
        return subprocess.Popen(
            self._build_command(),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
            text=True,
        )

    def _start_log_stream(self):
        """Stream server logs to the notebook cell."""
        def _reader():
            for line in self.server_process.stdout:
                print(f"[{self.port}] {line}", end="")

        threading.Thread(target=_reader, daemon=True).start()

    def _check_server_ready(self) -> None:
        """Check if server is alive using /v1/models."""
        url = f"{self.base_url}/models"
        headers = {"Authorization": f"Bearer {self.api_key}"}
        response = requests.get(url, headers=headers, timeout=5)
        response.raise_for_status()

    def wait_for_server(self):
        print(f"Waiting for vLLM server :{self.port} (GPUs {self.gpus})...")
        start = time.time()

        for _ in range(self.server_timeout):
            if self.server_process.poll() is not None:
                raise RuntimeError(f"vLLM server :{self.port} exited unexpectedly.")
            try:
                self._check_server_ready()
                print(f"Server :{self.port} ready in {time.time() - start:.2f}s\n")
                return
            except Exception:
                time.sleep(1)

        raise RuntimeError(f"Server :{self.port} failed to start (timeout).")

    def close(self):
        if hasattr(self, "server_process") and self.server_process.poll() is None:
            self.server_process.terminate()
            self.server_process.wait()

    def __del__(self):
        self.close()

In [7]:
def preload_weights(*model_dirs):
    """Warm the OS page cache by reading every shard in parallel across dirs.

    Overlaps disk I/O for both models up-front so each vLLM boot reads from
    RAM instead of disk. Pages are reclaimable: sleep-mode's CPU-side offload
    will evict them automatically under memory pressure.
    """
    procs = [
        subprocess.Popen(f'cat "{d}"/*.safetensors > /dev/null', shell=True)
        for d in model_dirs
    ]
    for proc in procs:
        proc.wait()
    print(f"Preloaded page cache for: {list(model_dirs)}")


preload_weights(VLLM_SERVER.model_path)

Preloaded page cache for: ['/kaggle/input/models/laureanoarcanio/swift-qwen3-8-27b-awq-int4/pytorch/default/1/Swift-Qwen3.8-27B-AWQ-INT4']


In [8]:
VLLM_SERVERS = [VLLM_SERVER(port=5001, gpus="0,1", wait=False),
                VLLM_SERVER(port=5002, gpus="2,3", wait=False)]
for server in VLLM_SERVERS:
    server.wait_for_server()

Waiting for vLLM server :5001 (GPUs 0,1)...
[5001] /usr/local/lib/python3.12/dist-packages/vllm/entrypoints/openai/api_server.py:24: DeprecationWarning: `vllm.entrypoints.openai.api_server` is deprecated and will likely beunsupported in a future version. Use the corresponding function from `vllm.entrypoints.launchers` instead.
[5001]   warnings.warn(
[5001] /usr/local/lib/python3.12/dist-packages/vllm/entrypoints/openai/api_server.py:51: DeprecationWarning: The `python -m vllm.entrypoints.openai.api_server` command is deprecated and may be removed in a future release. Please use `vllm server` instead.
[5001]   warnings.warn(
[5002] /usr/local/lib/python3.12/dist-packages/vllm/entrypoints/openai/api_server.py:24: DeprecationWarning: `vllm.entrypoints.openai.api_server` is deprecated and will likely beunsupported in a future version. Use the corresponding function from `vllm.entrypoints.launchers` instead.
[5002]   warnings.warn(
[5002] /usr/local/lib/python3.12/dist-packages/vllm/entryp

In [9]:
%%bash
for port in 5001 5002; do
curl http://localhost:$port/v1/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "qwen3.8",
    "prompt": "What is 2 + 2?",
    "max_tokens": 32,
    "temperature": 0.0
  }'
echo
done

[5001] (APIServer pid=37442) WARNING 10-16 22:21:30 [vllm.py:1327] Watermarking is enabled for this request, but the engine has no watermark configuration. This and subsequent requests will run without watermarking.
[5001] (Worker_TP0 pid=38640) WARNING 10-16 22:21:30 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _compute_local_logits_stats_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
[5001] (Worker_TP0 pid=38640) WARNING 10-16 22:21:30 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _rejection_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
[5001] (Worker_TP0 pid=38640) WARNING 10-16 22:21:30 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _resample_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
[5001] INFO:     127.0.0.1:53954 - "POST /v1/completions HTTP/1.1" 200 OK
[5002] (APIServer pid=

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   777  100   671  100   106    782    123 --:--:-- --:--:-- --:--:--   905
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   777  100   671  100   106    787    124 --:--:-- --:--:-- --:--:--   910


In [10]:
# ---- Configuration ----------------------------------------------------------
# Everything tunable for a run lives here, in the first cell of the notebook.
# The cell below only imports and derives from these.
import time
from pathlib import Path

# ---- Server -----------------------------------------------------------------
# One vLLM instance per URL. Every task is pinned to one of them (see ServerRouter), so a
# conversation never re-prefills its whole context on the other instance.
BASE_URLS = ["http://localhost:5001/v1", "http://localhost:5002/v1"]

SEED = 42
SEED_STEP = 10

SAMPLING_COUNT = 1

# QWEN
PARAMETERS = {
    "temperature": 1,
    "top_p": 0.95,
    "top_k": 20,
    "min_p": 0.0,
    "presence_penalty": 0,
    "repetition_penalty": 1.0,
}

PARAMETERS_SAMPLING = {
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "min_p": 0.0,
    "presence_penalty": 1.5,
    "repetition_penalty": 1.0,
}


TEMPLATE_KWARGS = {
    "reasoning_effort": "xhigh",
    "preserve_thinking": True,
}

CONCURRENCY = 16
MAX_TURNS = 200                  # agent turns per run
TOKEN_CAP = 96 * 1024          # cumulative generated tokens per run

PRIORITY_ENABLED = False
PRIORITY_CONFIG   = {
    "levels": [
        {"max_generated": 16_000, "priority": 0},
        {"max_generated": 24_000, "priority": 10},
        {"max_generated": 38_000, "priority": 20},
        {"max_generated": 64_000, "priority": 30},
    ],
    "default_priority": 40,
}

# Agent-loop guardrails (section 4). An unusable reply gets a retry message; a streak of
# them ends the run with no answer.
NO_CODE_ABORT_STREAK = 3            # replies in a row w/o usable code -> abort
TRUNCATED_NO_TOOL_ABORT_STREAK = 3  # truncated replies w/o tool call -> abort
CONTEXT_PAD = 512                   # tokens always left free by the max_tokens clamp

# Prompt-size estimate, in place of a /tokenize pre-flight (see Conversation). vLLM rejects
# a request whose prompt exceeds max_model_len - max_tokens instead of trimming it, so the
# estimate must never come in low. Charging one token per character makes that a guarantee
# rather than a guess: no token is shorter than a character. It is not the overshoot it
# looks like either - ARC prompts are grids of digits, where every digit and space is its
# own token, and measure ~1.15 chars/token.
PROMPT_CHARS_PER_TOKEN = 1.0        # a token is never less than one character
PROMPT_SLACK = 1.15                 # multiplier on everything appended since the anchor
PROMPT_MESSAGE_OVERHEAD = 16        # template tokens charged per appended message

EXEC_TIMEOUT_S = 20             # per code snippet

# Turns are streamed so the deadline can be enforced mid-generation: a turn runs tens of
# minutes and nothing interrupts it once started, so a whole turn's worth of overrun is
# what makes a run outlive the time budget. The clock is re-read every chunk.
#
# Nothing is ever cut for being slow or quiet, and there is no stall guard: CONCURRENCY is
# deliberately above the server's --max-num-seqs so the priority policy has a queue to
# reorder, and a request parked in that queue emits nothing until a slot frees - a wait
# that approaches a whole turn. Any silence threshold short enough to catch a wedged
# server is also short enough to kill those healthy runs at turn 0.
#
# So the read timeout is the deadline and nothing else. httpx measures time WITHOUT data
# and resets on every chunk, so the only way it can fire is that nothing at all arrived
# between the request and the deadline - at which point the run is over regardless.
STREAM_CONNECT_S = 300.0        # generous: the API server also serves every open stream

# "evaluation" grades against the known solutions; "test" is the competition set, which
# ships no solutions file - the run is then ungraded and the submission is the only output.
DATASET = "evaluation"
DATA_DIR = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
CHALLENGES_PATH = DATA_DIR / f"arc-agi_{DATASET}_challenges.json"
SOLUTIONS_PATH = DATA_DIR / f"arc-agi_{DATASET}_solutions.json"
LOG_DIR = Path("/kaggle/working/runs/local_agent_repl")
RUN_ID = ""                     # "" -> new run
SAVE_LOGS = True                # write per-turn debug logs under texts/

# ---- Task selection ---------------------------------------------------------
LOW_TASKS = [
    # Baseline sweep test set - Best 9/15 - worst 5/15
    "135a2760", "136b0064", "16de56c4", "1818057f", "1ae2feb7",
    "247ef758", "2ba387bc", "31f7f899", "35ab12c3", "36a08778",
    "38007db0", "45a5af55", "4a21e3da", "4c3d4a41", "53fb4810",

    # Full
    "58490d8a", "58f5dbd5", "5961cc34", "65b59efc", "67e490f4",
    "6e453dd6", "7491f3cf", "7b5033c1", "7c66cb00", "80a900e0",
    "89565ca0", "898e7135", "8e5c0c38", "8f215267", "8f3a5a89",
    "97d7923e", "981571dc", "9aaea919", "a251c730", "a395ee82",
    "a47bf94d", "a6f40cea", "aa4ec2a5", "b0039139", "b10624e5",
    # "b5ca7ac4", "b99e7126", "bf45cf4b", "c4d067a0", "d59b0160",
    # "d8e07eb2", "db0c5428", "db695cfb", "e8686506", "edb79dae",
    # "f931b4a8",
]

HIGH_TASKS = [
    "0934a4d8", "142ca369", "20270e3b", "221dfab4", "271d71e2",
    "291dc1e1", "2c181942", "409aa875", "64efde09", "71e489b6",
    "7666fa5d", "7b3084d4", "7ed72f31", "9385bd28", "b6f77b65",
    "cb2d8a2c", "cbebaa4b", "dbff022c", "dfadab01", "e376de54",
    "e87109e9", "fc7cae8d",
]

XHIGH_TASKS = [
    "195c6913", "28a6681f", "332f06d7", "3e6067c3", "5dbc8537",
    "6e4f6532", "7b0280bc", "8b9c3697", "c7f57c3e", "eee78d87",
]

TASK_IDS = LOW_TASKS             # [] -> every task in the challenge file

# ---- Run directory ----------------------------------------------------------
RUN_ID = RUN_ID or time.strftime("%Y%m%d-%H%M%S")
RUN_DIR = LOG_DIR / RUN_ID
TEXTS = RUN_DIR / "texts"
TEXTS.mkdir(parents=True, exist_ok=True)

# Kaggle scores submission.json in the notebook's WORKING DIRECTORY (/kaggle/working),
# not in the run directory - a submission written only under runs/ is invisible to the
# grader and the run scores 0. So the canonical file always goes to the working dir, and
# the run directory keeps a copy for the archive.
SUBMISSION_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
SUBMISSION_PATH = SUBMISSION_DIR / "submission.json"
SUBMISSION_ARCHIVE_PATH = RUN_DIR / "submission.json"


In [11]:
import json
import queue
import re
import shutil
import subprocess
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import NamedTuple

import httpx
import numpy as np
import requests
from openai import APIConnectionError, APITimeoutError, BadRequestError, OpenAI
from tqdm.auto import tqdm

clients = [OpenAI(base_url=url, api_key="none", max_retries=0,
                  timeout=httpx.Timeout(None, connect=STREAM_CONNECT_S))
           for url in BASE_URLS]
_model_cards = [requests.get(f"{url}/models", timeout=60).json()["data"][0]
                for url in BASE_URLS]
MODEL = _model_cards[0]["id"]
MAX_MODEL_LEN = min(card["max_model_len"] for card in _model_cards)

CHALLENGES = json.loads(CHALLENGES_PATH.read_text())
SOLUTIONS = json.loads(SOLUTIONS_PATH.read_text()) if SOLUTIONS_PATH.exists() else {}
TASK_IDS = list(TASK_IDS) or list(CHALLENGES)


def get_priority(generated, config=PRIORITY_CONFIG):
    """Priority for the next request of a run that has generated `generated` tokens."""
    for level in config["levels"]:
        if generated < level["max_generated"]:
            return level["priority"]
    return config["default_priority"]


def sample_seed(sample):
    return SEED + sample * SEED_STEP


def sample_params(sample):
    return PARAMETERS if sample == 0 else PARAMETERS_SAMPLING


CONFIG = {
    "run_id": RUN_ID, "model": MODEL, "base_urls": BASE_URLS, "seed": SEED,
    "seed_step": SEED_STEP, "sampling_count": SAMPLING_COUNT,
    "parameters": PARAMETERS, "parameters_sampling": PARAMETERS_SAMPLING,
    "template_kwargs": TEMPLATE_KWARGS, "concurrency": CONCURRENCY,
    "max_turns": MAX_TURNS, "token_cap": TOKEN_CAP,
    "no_code_abort_streak": NO_CODE_ABORT_STREAK,
    "truncated_no_tool_abort_streak": TRUNCATED_NO_TOOL_ABORT_STREAK,
    "context_pad": CONTEXT_PAD, "max_model_len": MAX_MODEL_LEN,
    "stream_connect_s": STREAM_CONNECT_S,
    "prompt_chars_per_token": PROMPT_CHARS_PER_TOKEN, "prompt_slack": PROMPT_SLACK,
    "prompt_message_overhead": PROMPT_MESSAGE_OVERHEAD,
    "exec_timeout_s": EXEC_TIMEOUT_S,
    "priority_enabled": PRIORITY_ENABLED, "priority_config": PRIORITY_CONFIG,
    "save_logs": SAVE_LOGS, "dataset": DATASET, "challenges": str(CHALLENGES_PATH),
    "graded": bool(SOLUTIONS), "submission": str(SUBMISSION_PATH), "task_ids": TASK_IDS,
    "time_budget_s": TIME_BUDGET_S, "time_margin_s": TIME_MARGIN_S,
    "run_start": RUN_START, "deadline": DEADLINE,
}
(RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")

print(f"model  {MODEL} @ {', '.join(BASE_URLS)}  (ctx {MAX_MODEL_LEN:,})")
print(f"run    {RUN_ID} -> {RUN_DIR}")
n_inputs = sum(len(CHALLENGES[pid]["test"]) for pid in TASK_IDS)
print(f"tasks  {len(TASK_IDS)} ({n_inputs} test inputs) from {CHALLENGES_PATH.name}"
      + ("" if SOLUTIONS else "  (no solutions file: run is ungraded)"))
print(f"budget {MAX_TURNS} turns, {TOKEN_CAP:,} tok/run, {SAMPLING_COUNT} samples/task"
      " | reasoning replayed on all turns")
deadline_at = time.strftime("%H:%M:%S", time.localtime(DEADLINE))
print(f"time   {time_left() / 3600:.2f}h until {deadline_at}")


[5001] INFO:     127.0.0.1:53964 - "GET /v1/models HTTP/1.1" 200 OK
[5002] INFO:     127.0.0.1:48222 - "GET /v1/models HTTP/1.1" 200 OK
model  qwen3.8 @ http://localhost:5001/v1, http://localhost:5002/v1  (ctx 98,304)
run    20261016-222132 -> /kaggle/working/runs/local_agent_repl/20261016-222132
tasks  40 (51 test inputs) from arc-agi_evaluation_challenges.json
budget 200 turns, 98,304 tok/run, 1 samples/task | reasoning replayed on all turns
time   11.88h until 10:14:07


In [12]:
AGENT_PROMPT_INTRO = (
    "You are participating in a puzzle solving competition. You are an expert at solving puzzles.\n"
    "Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern "
    "or transformation in the training examples that maps the input to the output, then apply that "
    "pattern to the test input and submit the resulting output grid.\n"
    "\n"
    "Grids are shown one row per line, cell values (0-9) space-separated, with the shape and a "
    "per-color cell count underneath - not as JSON. This is only the display format: submit() and "
    "the REPL still use plain lists of lists of ints.\n"
    "\n"
    "--Training Examples--"
)

AGENT_PROMPT_TRAIN_PAIR = (
    "\n--Example {n}-- \n"
    "\n INPUT: \n"
    "\n{input}\n"
    "\nOUTPUT: \n"
    "\n{output}\n"
)

AGENT_PROMPT_TEST = (
    "\n--End of Training Examples--\n"
    "\n--Test Input--\n"
    "\n{input}\n"
    "\n--End of Test Input--\n"
)

AGENT_PROMPT_TOOLS = """
You have a Python REPL, used through the python tool. These are already defined for you:

- train_inputs, train_outputs: the training grids above, as lists of lists of ints
- test_input: the test input grid
- np (numpy)
- show(grid): pretty-print a grid
- diff(a, b): print where two grids differ
- verify(fn): run fn on a copy of every training input and check the result against the
  training output; prints PASS/FAIL per pair and the diffs of failures
- submit(grid): submit the final output grid for the test input (this ends the task)
Note: the puzzle data (train_inputs, train_outputs, test_input(s)) is preloaded and READ-ONLY; the REPL blocks reassigning these names, so do not write e.g. train_inputs = [].

To run code, call the python tool with your code as the `code` argument. The execution
output comes back as the tool result. Variables persist between calls.

Rules:
1. Your first reply must call python to describe the input and test as needed.
2. Implement the transformation as a function taking an input grid and returning the output
   grid, and check it with verify(fn). It must reproduce every training output exactly.
3. Only call submit() after verify(fn) has passed on ALL training pairs in this session; the
   usual final step is submit(fn(test_input)). This applies even if you construct the output
   grid directly by hand: implement the transformation as a function and verify it first.
   Never submit a grid that is not backed by a passing verify(fn).
4. If verify(fn) fails, look at the failing pair with diff() and revise your hypothesis.
   Do not submit an unverified guess while you still have turns left.
"""

# Multi-test variant: same text with the test-input and submit lines adapted to a list of
# test inputs. Derived by replacement so the two versions cannot drift apart.
AGENT_PROMPT_TOOLS_MULTI = (
    AGENT_PROMPT_TOOLS
    .replace("- test_input: the test input grid",
             "- test_inputs: the list of ALL test input grids above, in order")
    .replace("- submit(grid): submit the final output grid for the test input (this ends the task)",
             "- submit(grids): submit the final output grids as a list, one grid per test input,\n"
             "  in the same order as above (this ends the task)")
    .replace("submit(fn(test_input))",
             "submit([fn(t) for t in test_inputs])")
)

NUDGE_MSG = (
    "No tool call found in your reply. Use the python tool to run code. "
    "To finish, call submit() from code."
)

BAD_ARGS_MSG = (
    "The tool call arguments were not valid JSON with a `code` string. "
    'Call python again with arguments like {"code": "..."}.'
)

CONTINUE_MSG = (
    "Your previous reply was truncated before a tool call. Do not continue the analysis "
    "in prose. Immediately call python with a small inspection or partial candidate. "
    "An incomplete experiment is better than more private deliberation."
)

# The single native tool: run Python in the per-run REPL. The chat template injects this
# schema, so the model emits proper tool calls instead of improvising a format. The Twu31
# Qwen3.x templates emit the qwen3-coder XML call format - the server must run with
# --enable-auto-tool-choice --tool-call-parser qwen3_coder (NOT hermes, which expects JSON
# inside <tool_call> and fails to parse this template's output).
TOOLS = [{
    "type": "function",
    "function": {
        "name": "python",
        "description": (
            "Run Python code in the persistent REPL for this task. The preloaded helpers "
            "(train_inputs, train_outputs, test_input(s), np, show, diff, verify, submit) "
            "are available and variables persist between calls. Returns the printed output."),
        "parameters": {
            "type": "object",
            "properties": {
                "code": {"type": "string", "description": "Python source code to execute"},
            },
            "required": ["code"],
        },
    },
}]


def describe_grid(grid):
    """Plain row-per-line text + shape + per-color cell counts, for prompt display.

    The model otherwise transcribes the grids by hand in its reasoning, reformatting the
    raw JSON into readable rows and counting colors. Both are pure transcription of data
    already in the prompt, at a smaller encoding than JSON besides.
    """
    height = len(grid)
    width = len(grid[0]) if grid else 0
    rows = "\n".join(" ".join(str(v) for v in row) for row in grid)
    counts = {}
    for row in grid:
        for value in row:
            counts[value] = counts.get(value, 0) + 1
    color_line = "colors: " + " ".join(f"{c}={counts[c]}" for c in sorted(counts))
    return f"{height}x{width}\n{rows}\n{color_line}"


def build_prompt(task, test_inputs):
    parts = [AGENT_PROMPT_INTRO]
    for i, pair in enumerate(task["train"]):
        parts.append(AGENT_PROMPT_TRAIN_PAIR.format(
            n=i, input=describe_grid(pair["input"]), output=describe_grid(pair["output"])))
    if len(test_inputs) == 1:
        parts.append(AGENT_PROMPT_TEST.format(input=describe_grid(test_inputs[0])))
        parts.append(AGENT_PROMPT_TOOLS)
    else:
        parts.append("\n--End of Training Examples--\n")
        for i, test_input in enumerate(test_inputs):
            parts.append(f"\n--Test Input {i}--\n\n{describe_grid(test_input)}\n")
        parts.append("\n--End of Test Inputs--\n")
        parts.append(AGENT_PROMPT_TOOLS_MULTI)
    return "".join(parts)


_CODE_RE = re.compile(r"```(?:python|py)?\s*\n(.*?)```", re.DOTALL)
_TOOLCALL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)
_XML_CODE_RE = re.compile(r"<parameter=code>\n?(.*?)\n?</parameter>", re.DOTALL)


def extract_calls(tool_calls, content):
    """List of (code, tool_call_id, source) entries from one chat response - one per
    native tool call, in order, ALL of which get executed. code is None with source
    "bad_args" for a native call whose arguments are not valid JSON with a `code` string
    (its id still needs a tool response). With no native calls, falls back to a single
    entry parsed from the visible content - qwen3-coder XML <parameter=code>,
    hermes-style JSON, or a plain ```python fence - or an empty list when nothing is
    usable.
    """
    if tool_calls:
        calls = []
        for call in tool_calls:
            code = None
            try:
                args = json.loads(call["function"]["arguments"])
            except ValueError:
                args = {}
            candidate = args.get("code")
            if isinstance(candidate, str) and candidate.strip():
                code = candidate.strip()
            calls.append((code, call["id"], "native" if code else "bad_args"))
        return calls

    match = _XML_CODE_RE.search(content or "")
    if match and match.group(1).strip():
        return [(match.group(1).strip(), None, "content")]

    for match in _TOOLCALL_RE.finditer(content or ""):
        try:
            payload = json.loads(match.group(1))
        except ValueError:
            continue
        code = (payload.get("arguments") or {}).get("code")
        if isinstance(code, str) and code.strip():
            return [(code.strip(), None, "content")]

    blocks = _CODE_RE.findall(content or "")
    if blocks and blocks[0].strip():
        return [(blocks[0].strip(), None, "fence")]
    return []


def grid_shape(grid):
    if not isinstance(grid, list) or not grid or not all(isinstance(r, list) for r in grid):
        return None
    widths = sorted({len(r) for r in grid})
    if len(widths) == 1:
        return f"{len(grid)}x{widths[0]}"
    return f"ragged ({len(grid)} rows, widths {'/'.join(map(str, widths))})"

In [13]:
SUBMISSION_ATTEMPTS = 2         # the competition scores at most two attempts per test input
BLANK_GRID = [[0, 0, 0], [0, 0, 0], [0, 0, 0]]

_submission_lock = threading.Lock()


def blank_submission(pids):
    """{task_id: [{"attempt_1": grid, "attempt_2": grid}, ...one per test input...]}."""
    return {
        pid: [{f"attempt_{k + 1}": [row[:] for row in BLANK_GRID]
               for k in range(SUBMISSION_ATTEMPTS)}
              for _ in CHALLENGES[pid]["test"]]
        for pid in pids
    }


def write_submission(submission):
    """Atomic write to the scored location, then mirror into the run directory. The temp
    file is created next to its destination so the replace() stays on one filesystem."""
    payload = json.dumps(submission)
    for path in (SUBMISSION_PATH, SUBMISSION_ARCHIVE_PATH):
        tmp = path.with_suffix(".json.tmp")
        tmp.write_text(payload, encoding="utf-8")
        tmp.replace(path)


def init_submission(pids):
    """A complete, valid submission from the start: all-black 3x3 placeholders that get
    overwritten as answers arrive, so an interrupted run still submits something. A file
    left by an earlier run of the same RUN_ID is kept (resume)."""
    submission = blank_submission(pids)
    resume_from = next((p for p in (SUBMISSION_ARCHIVE_PATH, SUBMISSION_PATH) if p.exists()), None)
    if resume_from is not None:
        try:
            prior = json.loads(resume_from.read_text())
        except ValueError:
            prior = {}
        for pid, entries in prior.items():
            if pid in submission and len(entries) == len(submission[pid]):
                submission[pid] = entries
    write_submission(submission)
    return submission


def update_submission(item):
    """Store a finished run's grids as attempt (sample + 1). Samples beyond the two
    scored attempts still land in results.jsonl, just not in the submission."""
    if item["sample"] >= SUBMISSION_ATTEMPTS:
        return
    attempt = f"attempt_{item['sample'] + 1}"
    with _submission_lock:
        entries = SUBMISSION[item["pid"]]
        for entry, answer in zip(entries, item["answers"]):
            if grid_shape(answer):
                entry[attempt] = answer
        write_submission(SUBMISSION)


SUBMISSION = init_submission(TASK_IDS)
print(f"submission -> {SUBMISSION_PATH} (copy: {SUBMISSION_ARCHIVE_PATH}) "
      f"({len(SUBMISSION)} tasks, {sum(len(v) for v in SUBMISSION.values())} test inputs)")

submission -> /kaggle/working/submission.json (copy: /kaggle/working/runs/local_agent_repl/20261016-222132/submission.json) (40 tasks, 51 test inputs)


In [14]:
CHILD = r"""
import ast, contextlib, dis, io, json, sys, traceback, types
import numpy as np

NS = {}
_RESERVED = frozenset({"verify", "submit", "submit_best", "show", "diff",
                       "train_inputs", "train_outputs", "test_inputs", "test_input",
                       "_last_verified_fn", "_last_verified_answers"})
_restore_helpers = lambda: None

def _reserved_bindings(tree):
    bad = set()
    root = compile(tree, "<agent>", "exec")
    def visit(code, module=False):
        for op in dis.get_instructions(code):
            writes_global = op.opname in {"STORE_GLOBAL", "DELETE_GLOBAL"}
            writes_module = module and op.opname in {"STORE_NAME", "DELETE_NAME"}
            if (writes_global or writes_module) and op.argval in _RESERVED:
                bad.add(op.argval)
        for value in code.co_consts:
            if isinstance(value, types.CodeType):
                visit(value)
    visit(root, module=True)
    return bad

def _reset(task):
    global _restore_helpers
    NS.clear()
    tri, tro, tests = task["train_inputs"], task["train_outputs"], task["test_inputs"]
    verified_answers = None

    def show(grid):
        for row in np.array(grid).tolist():
            print(" ".join(str(v) for v in row))

    def diff(a, b):
        A, B = np.array(a), np.array(b)
        if A.ndim != 2 or B.ndim != 2 or A.shape != B.shape:
            print(f"shape mismatch: {getattr(A, 'shape', '?')} vs {getattr(B, 'shape', '?')}")
            return
        d = np.argwhere(A != B)
        print(f"{len(d)}/{A.size} cells differ")
        for r, c in d[:20]:
            print(f"  ({r},{c}): {A[r, c]} vs {B[r, c]}")
        if len(d) > 20:
            print(f"  ... {len(d) - 20} more")

    def verify(fn):
        nonlocal verified_answers
        verified_answers = None
        all_ok = True
        for i, (inp, out) in enumerate(zip(tri, tro)):
            try:
                pred = np.array(fn([row[:] for row in inp])).tolist()
            except Exception as e:
                print(f"train {i}: ERROR {type(e).__name__}: {e}")
                all_ok = False
                continue
            if pred == out:
                print(f"train {i}: PASS")
            else:
                print(f"train {i}: FAIL")
                diff(out, pred)
                all_ok = False
        if not all_ok:
            print("verification FAILED - do not submit yet")
            return False
        try:
            verified_answers = [
                _check(fn([row[:] for row in test])) for test in tests]
        except Exception as e:
            print(f"verified fn failed on test input: {type(e).__name__}: {e}")
            return False
        print("ALL TRAIN PAIRS PASS")
        return True

    def _check(g):
        a = np.array(g)
        assert a.ndim == 2 and a.dtype.kind in "iu", "each grid must be a 2D list of ints"
        a = a.tolist()
        assert all(0 <= v <= 9 for row in a for v in row), "cell values must be 0-9"
        return a

    def _gridlike(g):
        if isinstance(g, np.ndarray):
            return g.ndim == 2
        return (isinstance(g, (list, tuple)) and len(g) > 0
                and isinstance(g[0], (list, tuple, np.ndarray)))

    def submit(answer):
        if verified_answers is None:
            print("submit blocked: call verify(fn) and obtain ALL TRAIN PAIRS PASS first")
            return
        # single test input: accept a bare grid (or a 1-list wrapping it);
        # several test inputs: require a list with exactly one grid per test input.
        if len(tests) == 1:
            a = np.array(answer)
            if a.ndim == 3 and a.shape[0] == 1:
                answer = a[0].tolist()
            grids = [_check(answer)]
        else:
            assert (isinstance(answer, (list, tuple)) and len(answer) == len(tests)
                    and all(_gridlike(g) for g in answer)), (
                f"submit expects a list of {len(tests)} grids, one per test input, in order")
            grids = [_check(g) for g in answer]
        if grids != verified_answers:
            print("submit blocked: answer differs from the last verified function; "
                  "call verify(fn) again or submit_best()")
            return
        print("__SUBMIT__" + json.dumps(grids))

    def submit_best():
        if verified_answers is None:
            print("no verified candidate available")
            return
        submit(verified_answers[0] if len(tests) == 1 else verified_answers)

    helpers = {"show": show, "diff": diff, "verify": verify,
               "submit": submit, "submit_best": submit_best}
    def restore_helpers():
        NS.update(helpers)
        NS.update(np=np, train_inputs=tri, train_outputs=tro,
                  test_inputs=tests, test_input=tests[0])
        NS.pop("_last_verified_fn", None)
        NS.pop("_last_verified_answers", None)
    _restore_helpers = restore_helpers
    NS.update(np=np, json=json,
              train_inputs=tri, train_outputs=tro,
              test_inputs=tests, test_input=tests[0])
    restore_helpers()

def _run(code):
    buf = io.StringIO()
    try:
        tree = ast.parse(code, "<agent>", "exec")
        bad = _reserved_bindings(tree)
        if bad:
            names = ", ".join(sorted(bad))
            buf.write(f"reserved name(s) cannot be assigned: {names}. "
                      "The puzzle data (train_inputs, train_outputs, test_input(s)) is "
                      "preloaded and read-only; rename your helper, for example "
                      "to check_candidate.")
        else:
            last = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None
            with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
                exec(compile(tree, "<agent>", "exec"), NS)
                if last is not None:
                    value = eval(compile(ast.Expression(last.value), "<agent>", "eval"), NS)
                    if value is not None:
                        print(repr(value), file=buf)
    except Exception:
        buf.write(traceback.format_exc(limit=3))
    finally:
        _restore_helpers()
    return buf.getvalue()

for line in sys.stdin:
    req = json.loads(line)
    if "task" in req:
        _reset(req["task"])
        out = "ready"
    else:
        out = _run(req["code"])
    print(json.dumps({"out": out}), flush=True)
"""


class Sandbox:
    """A persistent python REPL in a subprocess, one per agent run (task or test input)."""

    def __init__(self, task, test_inputs):
        self._payload = {"train_inputs": [p["input"] for p in task["train"]],
                         "train_outputs": [p["output"] for p in task["train"]],
                         "test_inputs": test_inputs}
        self._start()

    def _start(self):
        self._q = queue.Queue()
        self._proc = subprocess.Popen([sys.executable, "-u", "-c", CHILD],
                                      stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                      stderr=subprocess.DEVNULL, text=True)
        threading.Thread(target=self._reader, args=(self._proc,), daemon=True).start()
        self._rpc({"task": self._payload}, timeout=30)

    def _reader(self, proc):
        for line in proc.stdout:
            self._q.put(line)

    def _rpc(self, msg, timeout):
        self._proc.stdin.write(json.dumps(msg) + "\n")
        self._proc.stdin.flush()
        return json.loads(self._q.get(timeout=timeout))["out"]

    def run(self, code):
        """(submitted list of grids - one per test input - or None, output text)."""
        try:
            out = self._rpc({"code": code}, timeout=EXEC_TIMEOUT_S)
        except (queue.Empty, BrokenPipeError, OSError):
            self.close()
            self._start()
            return None, (f"Execution timed out after {EXEC_TIMEOUT_S}s and the REPL was "
                          "restarted. Your variables were lost; the preloaded ones are back. "
                          "Avoid loops over large ranges.")
        grids, kept = None, []
        for line in out.splitlines():
            if line.startswith("__SUBMIT__"):
                try:
                    grids = json.loads(line[len("__SUBMIT__"):])
                    kept.append("<grids submitted>")
                except Exception:
                    kept.append(line)
            else:
                kept.append(line)
        text = "\n".join(kept).strip() or "(no output)"
        return grids, text

    def close(self):
        try:
            self._proc.kill()
        except Exception:
            pass

In [15]:
GREEN, RED, YELLOW, RESET = "\033[32m", "\033[31m", "\033[33m", "\033[0m"


def fmt_duration(seconds):
    minutes, seconds = divmod(int(seconds), 60)
    return f"{minutes}m{seconds:02d}s" if minutes else f"{seconds}s"


class PriorityTracker:
    """Live count of in-flight requests per priority level. Printed with every turn so the
    effect of the vLLM priority scheduling is visible while the run goes on."""

    def __init__(self, config=PRIORITY_CONFIG):
        self._labels = {level["priority"]: f"<{level['max_generated']:,}"
                        for level in config["levels"]}
        self._labels.setdefault(config["default_priority"], "rest")
        self._counts = {priority: 0 for priority in self._labels}
        self._lock = threading.Lock()

    def enter(self, priority):
        with self._lock:
            self._counts[priority] = self._counts.get(priority, 0) + 1

    def leave(self, priority):
        with self._lock:
            self._counts[priority] -= 1

    def lines(self):
        with self._lock:
            counts = sorted(self._counts.items())
        return [f"{self._labels.get(p, str(p)):>9}: {n}" for p, n in counts]


PRIORITY_TRACKER = PriorityTracker()


class ServerStats:
    """The vLLM /metrics gauges that explain what the queue is doing, polled at most once
    a second and shared by every worker. Never raises: a scrape that fails just drops the
    line from the turn block."""

    def __init__(self, url, ttl=1.0):
        self._url = url
        self._ttl = ttl
        self._lock = threading.Lock()
        self._line = None
        self._at = 0.0
        self._capacity = None

    def line(self):
        with self._lock:
            if time.monotonic() - self._at > self._ttl:
                self._at = time.monotonic()
                self._line = self._scrape()
            return self._line

    def _scrape(self):
        try:
            raw = requests.get(self._url, timeout=2).text
        except requests.RequestException:
            return None
        if self._capacity is None:
            self._capacity = _metric_label(raw, "vllm:cache_config_info", "kv_cache_size_tokens")
        running = _metric(raw, "vllm:num_requests_running")
        waiting = _metric(raw, "vllm:num_requests_waiting")
        usage = _metric(raw, "vllm:kv_cache_usage_perc")
        hits = _metric(raw, "vllm:prefix_cache_hits_total")
        queries = _metric(raw, "vllm:prefix_cache_queries_total")
        preempt = _metric(raw, "vllm:num_preemptions_total")
        if running is None:
            return None
        parts = [f"running {running:.0f}  waiting {waiting or 0:.0f}"]
        if usage is not None:
            kv = f"kv {usage:.1%}"
            if self._capacity:
                kv += f" ({usage * self._capacity:,.0f}/{self._capacity:,.0f} tok)"
            parts.append(kv)
        if queries:
            parts.append(f"prefix hit {(hits or 0) / queries:.0%}")
        if preempt is not None:
            parts.append(f"preempt {preempt:.0f}")
        return "  ".join(parts)


def _metric(raw, name):
    match = re.search(rf"^{re.escape(name)}(?:\{{[^}}]*\}})? (\S+)$", raw, re.M)
    return float(match.group(1)) if match else None


def _metric_label(raw, name, label):
    match = re.search(rf'^{re.escape(name)}\{{[^}}]*{label}="([^"]*)"', raw, re.M)
    try:
        return int(match.group(1)) if match else None
    except ValueError:
        return None


SERVER_STATS = [ServerStats(f"{url.rsplit('/v1', 1)[0]}/metrics") for url in BASE_URLS]


class ServerRouter:
    """Pins each task to one vLLM instance for good: every turn of every sample of a task
    goes to the same server, so the prefix cache that holds its conversation is always the
    one it hits. A task is pinned when its first run starts, to the server with the fewest
    runs in flight (ties: fewest tasks pinned), which keeps the instances evenly loaded."""

    def __init__(self, n_servers):
        self._lock = threading.Lock()
        self._pinned = {}
        self._in_flight = [0] * n_servers
        self._tasks = [0] * n_servers

    def acquire(self, pid):
        with self._lock:
            server = self._pinned.get(pid)
            if server is None:
                server = min(range(len(self._in_flight)),
                             key=lambda s: (self._in_flight[s], self._tasks[s]))
                self._pinned[pid] = server
                self._tasks[server] += 1
            self._in_flight[server] += 1
            return server

    def release(self, server):
        with self._lock:
            self._in_flight[server] -= 1

    def line(self):
        with self._lock:
            return "  ".join(f"s{s}: {n} runs, {t} tasks"
                             for s, (n, t) in enumerate(zip(self._in_flight, self._tasks)))


ROUTER = ServerRouter(len(BASE_URLS))


class Reply(NamedTuple):
    content: str
    reasoning: str
    tool_calls: list
    tokens: int
    finish: str
    priority: object            # None when the turn was aborted (context_full)
    seconds: float
    prompt_tokens: int
    max_model_len: int


def chat(conversation, max_tokens, seed, params, level, server):
    """One chat completion, returned as a Reply. `level` is the scheduling priority the
    caller assigns to this turn (see get_priority); `server` is the index of the vLLM
    instance the task is pinned to (see ServerRouter).

    Native tool calling: TOOLS is declared to the chat template (the server needs
    --enable-auto-tool-choice --tool-call-parser qwen3_coder) and parsed tool calls come
    back normalized as plain dicts, ready to be replayed in the assistant message.

    The reply is streamed and the clock is re-read on every chunk, so the turn is cut the
    moment the time budget is gone instead of running to completion. Only the deadline ever
    cuts a turn - never slowness, and never silence, since a queued request is silent by
    definition (see STREAM_CONNECT_S). A cut turn is reported as finish "out_of_time" and
    ends the run.

    One request per turn: max_tokens is clamped from the conversation's prompt estimate,
    and usage from the stream's final chunk re-anchors it. A prompt with no room left, or
    an estimate that came up short and got the request rejected, ends the turn as
    context_full instead of raising.
    """
    started = time.perf_counter()
    estimate = conversation.prompt_estimate()
    max_tokens = min(max_tokens, MAX_MODEL_LEN - estimate - CONTEXT_PAD)
    if max_tokens < 256:
        return Reply("", "", [], 0, "context_full", None, time.perf_counter() - started,
                     estimate, MAX_MODEL_LEN)

    priority = level if PRIORITY_ENABLED else None    # tracked either way
    extra_body = {"top_k": params["top_k"], "min_p": params.get("min_p", 0.0),
                  "repetition_penalty": params.get("repetition_penalty", 1.0),
                  "chat_template_kwargs": TEMPLATE_KWARGS}
    if priority is not None:
        extra_body["priority"] = priority

    texts, thinks, partial_calls = [], [], {}
    usage = finish = None
    PRIORITY_TRACKER.enter(level)
    try:
        stream = clients[server].chat.completions.create(
            model=MODEL, messages=conversation.messages, max_tokens=max_tokens, seed=seed,
            temperature=params["temperature"], top_p=params["top_p"],
            presence_penalty=params.get("presence_penalty", 0.0), extra_body=extra_body,
            tools=TOOLS, tool_choice="auto", stream=True,
            stream_options={"include_usage": True},
            # a queued request yields nothing and so never reaches the clock check below;
            # the read timeout is what bounds it, and it is exactly the time left.
            timeout=httpx.Timeout(max(30.0, time_left()), connect=STREAM_CONNECT_S))
        with stream:                      # closing it disconnects, and vLLM aborts the request
            for chunk in stream:
                if chunk.usage:           # final chunk, carries the exact token counts
                    usage = chunk.usage
                if chunk.choices:
                    choice = chunk.choices[0]
                    finish = choice.finish_reason or finish
                    delta = choice.delta
                    if delta.content:
                        texts.append(delta.content)
                    extra = delta.model_extra or {}
                    think = extra.get("reasoning") or extra.get("reasoning_content")
                    if think:
                        thinks.append(think)
                    for call in delta.tool_calls or []:
                        slot = partial_calls.setdefault(call.index,
                                                        {"id": None, "name": None, "args": []})
                        if call.id:
                            slot["id"] = call.id
                        if call.function:
                            slot["name"] = call.function.name or slot["name"]
                            if call.function.arguments:
                                slot["args"].append(call.function.arguments)
                if out_of_time():
                    finish = "out_of_time"
                    break
    except BadRequestError:               # estimate came up short
        return Reply("", "", [], 0, "context_full", None, time.perf_counter() - started,
                     estimate, MAX_MODEL_LEN)
    except (APITimeoutError, APIConnectionError, httpx.RequestError):
        # httpx errors reach us raw: the SDK only wraps them around the initial request,
        # never around iterating the stream body (openai/_streaming.py::_iter_events).
        finish = "out_of_time" if out_of_time() else "stalled"
    finally:
        PRIORITY_TRACKER.leave(level)

    content, reasoning = "".join(texts), "".join(thinks)
    if "</think>" in content:             # no reasoning parser on the server: split ourselves
        think, content = content.split("</think>", 1)
        reasoning += think.replace("<think>", "")
    tool_calls = [{"id": slot["id"], "type": "function",
                   "function": {"name": slot["name"], "arguments": "".join(slot["args"])}}
                  for _, slot in sorted(partial_calls.items()) if slot["id"]]
    if usage:
        conversation.anchor_prompt_tokens(usage.prompt_tokens)
        tokens, prompt_tokens = usage.completion_tokens, usage.prompt_tokens
    else:                                 # cut before the final chunk: no exact counts
        tokens = (len(content) + len(reasoning)) // 4
        prompt_tokens = estimate
    return Reply(content.strip(), reasoning, tool_calls, tokens, finish or "out_of_time",
                 priority, time.perf_counter() - started, prompt_tokens, MAX_MODEL_LEN)


class Conversation:
    """The message list, the pending inputs the transcript logs need, and the running
    prompt-size estimate: an anchor (the exact usage.prompt_tokens the last completion
    reported) plus a generous charge for everything appended since."""

    def __init__(self, prompt):
        self.messages = [{"role": "user", "content": prompt}]
        self._pending = [("user", prompt)]
        self._anchor = 0            # exact prompt size, from the last completion
        self._new_tokens = 0        # generated tokens appended since
        self._new_chars = len(prompt)
        self._new_messages = 1

    def add_user(self, text):
        self.messages.append({"role": "user", "content": text})
        self._pending.append(("user", text))
        self._new_chars += len(text)
        self._new_messages += 1

    def add_tool(self, call_id, text):
        self.messages.append({"role": "tool", "tool_call_id": call_id, "content": text})
        self._pending.append(("tool", text))
        self._new_chars += len(text)
        self._new_messages += 1

    def answer_call(self, call_id, text):
        """A native tool call must always get its tool response or the template
        misrenders; content-parsed calls get a plain user message instead."""
        if call_id:
            self.add_tool(call_id, text)
        else:
            self.add_user(f"REPL output:\n```\n{text}\n```")

    def add_assistant(self, reply):
        """Reasoning must ride under "reasoning": vLLM silently drops "reasoning_content"
        on incoming messages."""
        message = {"role": "assistant", "content": reply.content}
        if reply.reasoning:
            message["reasoning"] = reply.reasoning
        if reply.tool_calls:
            message["tool_calls"] = reply.tool_calls    # every call is executed
        self.messages.append(message)
        self._new_tokens += reply.tokens        # content + reasoning + calls, exact
        self._new_messages += 1

    def take_pending(self):
        pending, self._pending = self._pending, []
        return pending

    def prompt_estimate(self):
        """Upper bound on the next prompt's token count (see PROMPT_SLACK)."""
        appended = self._new_tokens + self._new_chars / PROMPT_CHARS_PER_TOKEN
        return int(self._anchor + appended * PROMPT_SLACK
                   + PROMPT_MESSAGE_OVERHEAD * self._new_messages)

    def anchor_prompt_tokens(self, prompt_tokens):
        """Re-anchor on the exact size the server reported, so the guess never accumulates
        across turns."""
        self._anchor = prompt_tokens
        self._new_tokens = self._new_chars = self._new_messages = 0


def new_item(pid, sample):
    """One agent run: the whole task (all of its test inputs) at one sampling seed."""
    task = CHALLENGES[pid]
    truth = SOLUTIONS.get(pid)
    n_tests = len(task["test"])
    return {"pid": pid, "sample": sample, "seed": sample_seed(sample),
            "params": sample_params(sample), "label": f"{pid}/s{sample}",
            "task": task, "server": None,
            "test_inputs": [t["input"] for t in task["test"]],
            "refs": [truth[i] if truth else None for i in range(n_tests)],
            "log": [], "turns": 0, "generated": 0, "wall": 0.0, "finish": None,
            "done": False, "finished_at": None,
            "answers": [None] * n_tests, "answer_src": "none",
            "ok": False, "oks": [False] * n_tests, "cells": [None] * n_tests}


def cell_accuracy(pred, truth):
    if truth is None or grid_shape(pred) != grid_shape(truth):
        return None
    return float((np.array(pred) == np.array(truth)).mean())


def grade(item):
    item["oks"] = [r is not None and a == r for a, r in zip(item["answers"], item["refs"])]
    item["ok"] = bool(item["oks"]) and all(item["oks"])
    item["cells"] = [cell_accuracy(a, r) for a, r in zip(item["answers"], item["refs"])]


def item_dir(item, suffix="_in_progress"):
    return TEXTS / f"{item['pid']}-s{item['sample']}{suffix}"


def save_turn(item, name, text):
    """One file per turn, written as the turn completes. item["log"] always keeps the
    text in memory; files obey SAVE_LOGS."""
    item["log"].append(text)
    if not SAVE_LOGS:
        return
    directory = item_dir(item)
    directory.mkdir(parents=True, exist_ok=True)
    (directory / f"{name}.txt").write_text(text, encoding="utf-8")


def finish_log(item):
    """_in_progress -> _correct/_incorrect. Crashed/interrupted items keep _in_progress."""
    if not SAVE_LOGS:
        return
    src = item_dir(item)
    dst = item_dir(item, "_correct" if item["ok"] else "_incorrect")
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        src.rename(dst)


def report_turn(item, label, reply):
    """Account the turn and print it with a snapshot of the priority levels in flight."""
    item["generated"] += reply.tokens
    item["finish"] = reply.finish
    priority = "-" if reply.priority is None else str(reply.priority)
    context = ""
    if reply.max_model_len:
        context = (f" - ctx {reply.prompt_tokens:,}/{reply.max_model_len:,} "
                   f"({reply.prompt_tokens / reply.max_model_len:.0%})")
    block = [f"== {item['label']} @ server {item['server']} ==",
             f"{label} - prio {priority} - {reply.tokens:,} tok - "
             f"{item['generated']:,} total{context} - {reply.seconds:.1f}s turn - "
             f"{fmt_duration(time.perf_counter() - item['t0'])} total",
             "== PRIORITY (tokens generated) =="]
    block += PRIORITY_TRACKER.lines()
    block += ["== ROUTER ==", ROUTER.line()]
    server = SERVER_STATS[item["server"]].line()
    if server:
        block += [f"== SERVER {item['server']} ==", server]
    block.append("=" * 14)
    tqdm.write("\n".join(block))


def turn_report(conversation, reply):
    """The transcript section for one turn: the inputs it consumed, then the reply."""
    parts = [f"### input [{role}]\n{text}" for role, text in conversation.take_pending()]
    parts.append(f"### reasoning ({reply.tokens} tok, finish={reply.finish}, "
                 f"{reply.seconds:.1f}s)\n{reply.reasoning}")
    parts.append(f"### reply\n{reply.content}")
    return parts


def run_calls(conversation, sandbox, calls, parts):
    """Run every tool call in order and feed each result back. Returns the submitted
    grids as soon as one call submits (later calls are moot), otherwise None."""
    for code, call_id, source in calls:
        if code is None:
            parts.append("### tool call (bad_args)\n(unparsable arguments)")
            if call_id:
                conversation.add_tool(call_id, BAD_ARGS_MSG)
            continue
        grids, output = sandbox.run(code)
        parts.append(f"### tool call ({source})\n{code}")
        parts.append(f"### tool result\n{output}")
        if grids is not None:
            if call_id:
                conversation.add_tool(call_id, "<grids submitted>")
            return grids
        conversation.answer_call(call_id, output)
    return None


def agent_loop(item, conversation, sandbox):
    """Chat turn -> run the code -> feed the output back, until submit() fires or a
    budget (turns, tokens, wall clock) runs out. A real submit() is the only thing that
    sets item["answers"]/["answer_src"]; every other exit ends the run without an answer.
    item["finish"] records which budget or stall ended it."""
    no_code_streak = 0
    truncated_no_tool_streak = 0

    while item["turns"] < MAX_TURNS:
        if out_of_time():
            item["finish"] = "out_of_time"
            return
        room = TOKEN_CAP - item["generated"]
        if room < 1_000:
            item["finish"] = "token_cap"
            return

        reply = chat(conversation, room, item["seed"], item["params"],
                     get_priority(item["generated"]), item["server"])
        item["turns"] += 1
        report_turn(item, f"turn {item['turns']:2d}", reply)
        parts = turn_report(conversation, reply)
        turn_name = f"turn_{item['turns'] - 1}"
        if reply.finish in ("context_full", "out_of_time", "stalled"):
            item["finish"] = reply.finish
            parts.append(f"### aborted\n{reply.finish}")
            save_turn(item, turn_name, "\n\n".join(parts))
            return
        conversation.add_assistant(reply)

        calls = extract_calls(reply.tool_calls, reply.content)
        runnable = [c for c in calls if c[0] is not None]

        if not runnable and reply.finish == "length":
            # Truncated mid-thought. Two consecutive long thinks can be legitimate deep
            # work, so only a streak counts as a stall.
            truncated_no_tool_streak += 1
            for _, call_id, _ in calls:      # clipped native calls still need answers
                conversation.answer_call(call_id, BAD_ARGS_MSG)
            parts.append("### tool call\n(turn truncated without a usable call)")
            save_turn(item, turn_name, "\n\n".join(parts))
            if truncated_no_tool_streak >= TRUNCATED_NO_TOOL_ABORT_STREAK:
                item["finish"] = "truncated_stall"
                return
            if not calls:
                conversation.add_user(CONTINUE_MSG)
            continue

        if not runnable:
            no_code_streak += 1
            nudging = no_code_streak < NO_CODE_ABORT_STREAK
            for _, call_id, _ in calls:      # native calls with bad args: answer each id
                conversation.answer_call(call_id, BAD_ARGS_MSG if nudging else "(not executed)")
            if not calls and nudging:
                conversation.add_user(NUDGE_MSG)
            parts.append("### tool call\n(no usable code"
                         + (" - nudged)" if nudging else " - aborting)"))
            save_turn(item, turn_name, "\n\n".join(parts))
            if not nudging:
                item["finish"] = "no_code_stall"
                return                       # stopped using the REPL: give up
            continue

        no_code_streak = 0
        truncated_no_tool_streak = 0
        grids = run_calls(conversation, sandbox, calls, parts)
        save_turn(item, turn_name, "\n\n".join(parts))
        if grids is not None:
            item["answers"], item["answer_src"] = grids, "submit"
            return

    item["finish"] = "max_turns"


def run_item(item):
    """One agent run end to end. Never raises for a lack of answer: an item with no grids
    keeps its blank submission placeholders."""
    item["t0"] = time.perf_counter()
    if out_of_time():
        item["finish"] = "out_of_time"
        return item
    sandbox = Sandbox(item["task"], item["test_inputs"])
    conversation = Conversation(build_prompt(item["task"], item["test_inputs"]))
    item["server"] = ROUTER.acquire(item["pid"])
    try:
        agent_loop(item, conversation, sandbox)
    finally:
        ROUTER.release(item["server"])
        sandbox.close()
        item["done"] = True
        item["wall"] = time.perf_counter() - item["t0"]
        item["finished_at"] = time.time() - RUN_START      # orders the results table
        grade(item)
        finish_log(item)
        update_submission(item)
    return item

In [ ]:
RESULTS_PATH = RUN_DIR / "results.jsonl"


def mark(item):
    return f"{GREEN}✓{RESET}" if item["ok"] else f"{RED}✗{RESET}"


def pct(item):
    values = [c for c in item["cells"] if c is not None]
    return f"{sum(values) / len(values):5.0%}" if values else "  -  "


def write_results(items):
    """One row per (task, test input, sample). Cost fields cover the whole agent run and
    are attributed to its first row, so sums over rows stay correct."""
    rows = []
    for item in items:
        for i in range(len(item["test_inputs"])):
            rows.append({"task_id": item["pid"], "test_index": i, "sample": item["sample"],
                         "seed": item["seed"], "server": item["server"],
                         "done": item["done"],
                         "correct": item["oks"][i], "cell_accuracy": item["cells"][i],
                         "answer_src": item["answer_src"],
                         "answer_shape": grid_shape(item["answers"][i]),
                         "truth_shape": grid_shape(item["refs"][i]),
                         "turns": item["turns"], "finished_at": item["finished_at"],
                         "generated": item["generated"] if i == 0 else 0,
                         "finish": item["finish"],
                         "wall_s": round(item["wall"], 1) if i == 0 else 0,
                         "answer_grid": item["answers"][i], "truth_grid": item["refs"][i]})
    RESULTS_PATH.write_text("".join(json.dumps(r) + "\n" for r in rows), encoding="utf-8")
    return rows


def task_status(rows):
    """(correct, closed, total) task counts. A task counts as correct when any one sample
    got every test input right - the same best-of-attempts rule as the competition."""
    by_task = {}
    for row in rows:
        by_task.setdefault(row["task_id"], {}).setdefault(row["sample"], []).append(row)
    closed = correct = 0
    for samples in by_task.values():
        finished = [rs for rs in samples.values() if all(r["done"] for r in rs)]
        if not finished:
            continue
        closed += 1
        correct += any(all(r["correct"] for r in rs) for rs in finished)
    return correct, closed, len(by_task)


def restore_item(item, rows_by_key):
    """Rebuild a finished item from results.jsonl so a resumed run keeps its rows."""
    keys = [(item["pid"], i, item["sample"]) for i in range(len(item["test_inputs"]))]
    if not all(k in rows_by_key for k in keys):
        return False
    rows = [rows_by_key[k] for k in keys]
    if not all(r["done"] for r in rows):
        return False
    first = rows[0]
    item.update(done=True, turns=first["turns"], generated=first["generated"],
                wall=first["wall_s"], finish=first["finish"],
                finished_at=first.get("finished_at"),
                answer_src=first["answer_src"],
                answers=[r["answer_grid"] for r in rows],
                refs=[r["truth_grid"] for r in rows])
    grade(item)
    return True


# Sample-major order: every task gets its first attempt before any second one starts, so
# a run cut short by the time budget still has one answer per task.
items = [new_item(pid, sample)
         for sample in range(SAMPLING_COUNT) for pid in TASK_IDS]

done_rows = {}
if RESULTS_PATH.exists():
    for line in RESULTS_PATH.read_text().splitlines():
        row = json.loads(line)
        done_rows[(row["task_id"], row["test_index"], row["sample"])] = row
resumed = [it for it in items if restore_item(it, done_rows)]
to_run = [it for it in items if not it["done"]]
if resumed:
    print(f"resume: {len(resumed)}/{len(items)} agent runs already done in {RUN_ID}")

print(f"{len(TASK_IDS)} tasks x {SAMPLING_COUNT} samples -> {len(items)} agent runs "
      f"({len(to_run)} to run), concurrency {CONCURRENCY}, "
      f"{time_left() / 3600:.2f}h of budget left")

bar = tqdm(total=len(to_run), unit="run", dynamic_ncols=True,
           bar_format="{desc} |{bar}| {n_fmt}/{total_fmt} runs [{elapsed}<{remaining}]{postfix}")
t0 = time.time()
correct = sum(bool(it["ok"]) for it in resumed)
tokens = sum(it["generated"] for it in resumed)
skipped = 0
with ThreadPoolExecutor(CONCURRENCY) as pool:
    futures = {pool.submit(run_item, it): it for it in to_run}
    for future in as_completed(futures):
        item = futures[future]
        try:
            future.result()
        except Exception as exc:
            bar.write(f"  {RED}crashed{RESET} {item['label']}  {exc!r}")
        if not item["done"]:
            skipped += 1                    # never started: out of time
        else:
            correct += bool(item["ok"])
            tokens += item["generated"]
            bar.write(f"{mark(item)} {item['label']}  {item['turns']:2d} turns  "
                      f"{item['generated']:>7,} tok  cells {pct(item)}  "
                      f"via {item['answer_src']}")
        bar.update(1)
        bar.set_postfix_str(f"ok {correct}/{bar.n} | {tokens / 1000:.0f}k tok | "
                            f"{tokens / max(time.time() - t0, 1e-9):.1f} tok/s")
        write_results(items)
bar.close()

rows = write_results(items)
good, closed, n_tasks = task_status(rows)
if skipped:
    print(f"{YELLOW}time budget reached: {skipped} runs skipped{RESET}")
print(f"tasks {good}/{closed} correct of {n_tasks} | "
      f"{tokens / 1000:.1f}k tokens in {(time.time() - t0) / 60:.1f} min | "
      f"submission {SUBMISSION_PATH}")

40 tasks x 1 samples -> 40 agent runs (40 to run), concurrency 16, 11.88h of budget left


 |          | 0/40 runs [00:00<?]

[5002] INFO:     127.0.0.1:48226 - "POST /v1/chat/completions HTTP/1.1" 200 OK
[5001] INFO:     127.0.0.1:53970 - "POST /v1/chat/completions HTTP/1.1" 200 OK
[5002] INFO:     127.0.0.1:48240 - "POST /v1/chat/completions HTTP/1.1" 200 OK
[5001] INFO:     127.0.0.1:53976 - "POST /v1/chat/completions HTTP/1.1" 200 OK
[5002] INFO:     127.0.0.1:48244 - "POST /v1/chat/completions HTTP/1.1" 200 OK
[5001] INFO:     127.0.0.1:53990 - "POST /v1/chat/completions HTTP/1.1" 200 OK
[5002] INFO:     127.0.0.1:48270 - "POST /v1/chat/completions HTTP/1.1" 200 OK
[5001] INFO:     127.0.0.1:54006 - "POST /v1/chat/completions HTTP/1.1" 200 OK
[5002] INFO:     127.0.0.1:48258 - "POST /v1/chat/completions HTTP/1.1" 200 OK
[5002] INFO:     127.0.0.1:48280 - "POST /v1/chat/completions HTTP/1.1" 200 OK
[5002] INFO:     127.0.0.1:48294 - "POST /v1/chat/completions HTTP/1.1" 200 OK
[5001] INFO:     127.0.0.1:53992 - "POST /v1/chat/completions HTTP/1.1" 200 OK
[5002] INFO:     127.0.0.1:48310 - "POST /v1/chat/co

In [ ]:
# One row per task, in the order the tasks finished. The cost columns sum every sample; a
# sample counts as correct only when all of the task's test inputs are right - the rule the
# competition scores an attempt by - and the task counts when any sample manages it.
def sample_ok(sample_rows):
    return bool(sample_rows) and all(r["correct"] for r in sample_rows)


def tick(value):
    if value is None:
        return " -  "                    # sample never ran
    return f" {GREEN}\u2713{RESET}  " if value else f" {RED}\u2717{RESET}  "


by_task = {}
for row in rows:
    by_task.setdefault(row["task_id"], {}).setdefault(row["sample"], []).append(row)


def finished_at(pid):
    """When the task's last sample came back. Unfinished samples sort to the end."""
    return max((r["finished_at"] if r["finished_at"] is not None else float("inf"))
               for sample_rows in by_task[pid].values() for r in sample_rows)


print(f"{'task_id':<10}{'turns':>6}{'tokens':>10}{'time':>9}  "
      + "".join(f"s{k:<3}" for k in range(SAMPLING_COUNT)) + " ok")
totals = {"turns": 0, "tokens": 0, "secs": 0.0}
solved = ran = 0
for pid in sorted(by_task, key=finished_at):
    samples = by_task[pid]
    task_rows = [r for sample_rows in samples.values() for r in sample_rows]
    turns = sum(sample_rows[0]["turns"] for sample_rows in samples.values())
    tokens = sum(r["generated"] for r in task_rows)      # only the first row of a run carries these
    secs = sum(r["wall_s"] for r in task_rows)
    flags = [sample_ok(samples[k]) if k in samples and any(r["done"] for r in samples[k]) else None
             for k in range(SAMPLING_COUNT)]
    ok = any(bool(f) for f in flags)
    if any(f is not None for f in flags):
        ran += 1
        solved += ok
        totals["turns"] += turns
        totals["tokens"] += tokens
        totals["secs"] += secs
    print(f"{pid:<10}{turns:>6}{tokens:>10,}{fmt_duration(secs):>9}  "
          + "".join(tick(f) for f in flags) + tick(ok).rstrip())

print(f"\n{solved}/{ran} tasks correct (at least one sample) of {len(by_task)} total")
print(f"{totals['turns']:,} turns   {totals['tokens']:,} tokens   "
      f"{fmt_duration(totals['secs'])} of agent time")
print(f"elapsed {(time.time() - RUN_START) / 3600:.2f}h of the {TIME_BUDGET_S / 3600:.0f}h budget"
      f"  ({time_left() / 60:.1f} min left)")
print(f"results: {RESULTS_PATH}")
print(f"submission: {SUBMISSION_PATH}  ({SUBMISSION_PATH.stat().st_size} bytes)"
      if SUBMISSION_PATH.exists() else f"{RED}submission MISSING at {SUBMISSION_PATH}{RESET}")
print(f"submission copy: {SUBMISSION_ARCHIVE_PATH}")
print(f"transcripts: {TEXTS}")
